In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beijing")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beijing


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_Beijing_before.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
SQ0001,5.49,2.46,0.022
SQ0002,5.27,4.61,0.020
sQ2177,3.85,3.25,0.018
sQ2177A,2.99,1.92,0.012
sQ2178,2.73,0.98,0.011


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_Beijing_imp1.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
SQ0001,4.98,2.49,0.019
SQ0002,4.78,4.58,0.019
sQ2177,3.29,3.31,0.015
sQ2177A,2.32,1.90,0.010
sQ2178,2.14,0.92,0.008


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
BeijingPumpStation,3.36,8.76,0.523,3.06,11.29,0.561,-0.038,-7.3,0.30,8.9
sQ3AC,22.97,11.39,0.068,22.97,11.30,0.068,0.000,0.0,0.00,0.0
sQ3AC1,6.58,7.26,0.063,6.21,7.25,0.063,0.000,0.0,0.37,5.6
sQ3AB,5.13,5.15,0.048,4.77,5.15,0.047,0.001,2.1,0.36,7.0
sQ3AA,4.39,2.53,0.026,3.94,2.51,0.024,0.002,7.7,0.45,10.3
sQ3B,5.05,2.77,0.024,4.56,2.91,0.020,0.004,16.7,0.49,9.7
sQ3C,5.33,2.56,0.023,4.83,2.67,0.020,0.003,13.0,0.50,9.4
sQ3D,5.43,2.59,0.023,4.92,2.60,0.020,0.003,13.0,0.51,9.4
SQ0001,5.49,2.46,0.022,4.98,2.49,0.019,0.003,13.6,0.51,9.3


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# --- Duration ---
network_pre_hours = comparison["Pre Hours Flooded"].sum()
network_post_hours = comparison["Post Hours Flooded"].sum()
network_hours_reduction = network_pre_hours - network_post_hours

network_hours_percent_reduction = np.where(
    network_pre_hours > 0,
    (network_hours_reduction / network_pre_hours) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flood Duration ---")
print(f"Pre-Project Total Hours Flooded: {network_pre_hours:.2f} hrs")
print(f"Post-Project Total Hours Flooded: {network_post_hours:.2f} hrs")
print(f"Total Hours Flooded Reduction: {network_hours_reduction:.2f} hrs")
print(f"Hours Flooded Percent Reduction: {network_hours_percent_reduction:.1f}%")

Pre-Project Network Total Flood Volume: 1.042 MG
Post-Project Network Total Flood Volume: 1.012 MG
Network Flood Volume Reduction: 0.030 MG
Network Flood Volume Percent Reduction: 2.9%

--- Flood Duration ---
Pre-Project Total Hours Flooded: 132.35 hrs
Post-Project Total Hours Flooded: 119.53 hrs
Total Hours Flooded Reduction: 12.82 hrs
Hours Flooded Percent Reduction: 9.7%


In [8]:
# Fix: ensure these are scalars (not numpy arrays)
network_percent_reduction = float(network_percent_reduction)
network_hours_percent_reduction = float(network_hours_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)",
        "Pre-Project Total Hours Flooded (hrs)",
        "Post-Project Total Hours Flooded (hrs)",
        "Total Hours Flooded Reduction (hrs)",
        "Hours Flooded Percent Reduction (%)"
    ],
    "Value": [
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%",
        round(network_pre_hours, 2),
        round(network_post_hours, 2),
        round(network_hours_reduction, 2),
        f"{round(network_hours_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")

PermissionError: [Errno 13] Permission denied: 'network_flood_summary.csv'